# - Notebook projet BI - 

## Problématique 

**Comment structurer et exploiter les données du Pokémon TCG afin d’obtenir une vision complète, fiable et historisée du catalogue de cartes ?**

Le projet vise à construire un socle décisionnel permettant d’explorer et de suivre le catalogue du Pokémon Trading Card Game à partir d’une seule source de départ : l’API TCGdex. L’objectif n’est pas de prédire un phénomène externe, mais de transformer des données techniques et descriptives en une information structurée, historisée et exploitable dans un tableau de bord.

La problématique retenue est : « Comment structurer et exploiter les données du Pokémon TCG afin d’obtenir une vision complète, fiable et historisée du catalogue de cartes ? »

Cette problématique présente un intérêt managérial car un catalogue de cartes est volumineux et multidimensionnel. Une lecture directe de l’API ne facilite pas nécessairement les comparaisons entre séries, extensions et cartes. Le datawarehouse permet donc de centraliser les informations, de contrôler leur qualité, de conserver les différentes observations reçues au fil des chargements et de fournir une couche Gold stable pour le reporting.

Le dashboard doit notamment permettre de répondre à des questions telles que : quelles séries et extensions composent le catalogue ? combien de cartes sont présentes par extension ? quelle est la répartition des cartes par rareté, catégorie, type ou niveau d’évolution ? quelles cartes sont actuellement légales dans les formats disponibles ? quelles caractéristiques ont changé entre deux chargements ?

Adresse de la doc de l'API: [https://tcgdex.dev/](https://tcgdex.dev/)

Adresse de l'API: [https://api.tcgdex.net/v2/en/cards/swsh3-136](https://api.tcgdex.net/v2/en/cards/swsh3-136)


## Source de données et architecture

La source de départ est l’API REST TCGdex. La documentation indique que l’API fournit des réponses JSON et des endpoints consacrés aux cartes, aux sets et aux séries. 
Les informations détaillées d’une carte comprennent notamment son identifiant, son nom, son illustrateur, sa rareté, son set, ses variantes, ses points de vie, ses types, ses attaques, ses faiblesses, sa retraite, sa marque de régulation et ses informations de légalité. 
Les sets fournissent notamment leur date de sortie, leur série, leur statut de légalité et leurs compteurs de cartes. Les séries regroupent les sets qui leur sont associés.

L’architecture retenue suit une logique Medallion en trois couches :

• Bronze : conservation des réponses JSON brutes de TCGdex, avec un identifiant de chargement, une date/heure d’ingestion et les métadonnées techniques nécessaires au rejeu.

• Silver : données nettoyées, typées, dédupliquées et normalisées. Les objets imbriqués comme les attaques, types, faiblesses et variantes sont séparés lorsque cela facilite l’analyse.

• Gold : modèle dimensionnel destiné au reporting Power BI. Cette couche ne dépend que des données préparées provenant de TCGdex.

Cette séparation respecte le principe de conservation du brut et de reconstruction depuis les couches amont : une correction dans Gold ne doit pas devenir la seule source de vérité. La Bronze permet de rejouer un chargement sans devoir solliciter à nouveau la source pour la période conservée.


## Modèle dimensionnel Gold

La Gold est conçue pour être complète du point de vue des informations réellement disponibles dans TCGdex. Elle ne nécessite aucune donnée externe.

Les principales tables sont :

DIM_DATE : dimension calendrier utilisée pour analyser les chargements et les dates de publication des sets. Clé primaire : date_key.

DIM_SERIE : une ligne par série TCGdex. Clé primaire : serie_key. Attributs principaux : serie_id, nom, logo.

DIM_SET : une ligne par extension/set. Clé primaire : set_key. Attributs principaux : set_id, nom, serie_key, release_date, logo, symbole, nombre officiel de cartes, nombre total de cartes, légalité Standard et Expanded.

DIM_CARD_SCD2 : dimension historisée des cartes. Clé primaire technique : card_key. La business key est card_id. Attributs : nom, catégorie, illustrateur, rareté, set_id, local_id, HP, stage, evolve_from, regulation_mark, legal_standard, legal_expanded, variantes, image, valid_from, valid_to, is_current.

FACT_CARD_SNAPSHOT : table de faits au grain « une carte observée lors d’un chargement ». Clé primaire technique : snapshot_key. Clés étrangères : date_key, card_key, set_key, serie_key. Mesures/indicateurs : nombre de types, nombre d’attaques, nombre de faiblesses, nombre d’éléments de retraite, indicateurs de présence d’attaques et de variantes, et statut de complétude de l’observation.

Le grain de la fact est volontairement explicite : une même carte peut apparaître dans plusieurs snapshots, ce qui permet de comparer les états du catalogue entre les chargements.


## Historisation SCD Type 2

L’historisation est réalisée dans DIM_CARD_SCD2. Elle répond à l’exigence du travail concernant le suivi des évolutions.

Lorsqu’une carte est chargée pour la première fois, une version est créée avec valid_from égal à la date du chargement, valid_to à une date de fin conventionnelle et is_current à true. Lors d’un chargement ultérieur, si les attributs historisés sont inchangés, aucune nouvelle version n’est créée. Si un attribut change, l’ancienne version est clôturée et une nouvelle version est créée.

La comparaison doit porter sur les attributs pertinents et non sur les seuls champs techniques d’ingestion. Une clé de hachage des attributs historisés peut être utilisée pour faciliter la détection de changement.

Exemple : si la description d’une carte, sa rareté, sa légalité ou une autre caractéristique exposée par TCGdex est modifiée entre deux chargements, le modèle conserve les deux états. Le reporting peut alors distinguer la situation actuelle de la situation observée lors d’un chargement précédent.

Le contrôle qualité doit vérifier qu’une business key ne possède pas deux versions courantes et que les périodes de validité d’une même carte ne se chevauchent pas.


## Chargements multiples et reload

Le projet prévoit au minimum trois chargements afin de démontrer la logique d’orchestration et d’historisation demandée par les consignes.

Load 1 constitue le point de départ du catalogue. Load 2 est exécuté à une date ultérieure et Load 3 encore après. Chaque chargement conserve les données reçues dans Bronze. Même si TCGdex ne fournit pas un flux métier d’événements, ces snapshots successifs permettent d’observer si le contenu exposé par l’API a évolué.

Le reload concerne FACT_CARD_SNAPSHOT. Le principe consiste à supprimer/reconstruire un périmètre de snapshots déterminé, puis à recalculer la fact à partir des données Bronze conservées. La dimension SCD2 est ainsi évitée comme dépendance directe du reload de la fact, conformément à la recommandation des consignes.

Le pipeline doit être idempotent : relancer le même lot ne doit pas créer de doublons. Une clé de déduplication combinant card_id et load_id peut être utilisée au niveau des snapshots.


## Orchestration dans Databricks

Le pipeline Databricks est organisé en étapes dépendantes :

1.	ingestion TCGdex → Bronze ;
2.	contrôles techniques du lot ;
3.	transformation Bronze → Silver ;
4.	détection des changements et mise à jour SCD2 ;
5.	alimentation de FACT_CARD_SNAPSHOT ;
6.	contrôles Gold ;
7.	publication des tables destinées à Power BI.

Les étapes peuvent être organisées dans un Databricks Job avec des dépendances explicites. Chaque exécution doit conserver son statut, ses horaires, son identifiant de lot et des métriques simples : lignes reçues, lignes rejetées, lignes dédupliquées et lignes publiées.

La fréquence choisie doit rester cohérente avec la nature de la source. Ici, un traitement batch est adapté : le besoin porte sur la consolidation et le suivi d’un catalogue, pas sur une réaction seconde par seconde. La fréquence exacte sera définie dans l’implémentation en fonction des limites et de la stabilité observées sur l’API.


## Reporting Power BI

Power BI se connecte à la Gold en mode Import, conformément aux consignes du travail.

Le dashboard peut être organisé en trois vues.
- Vue « Catalogue » : nombre de séries, nombre de sets, nombre de cartes, répartition par série, set, catégorie et rareté.
- Vue « Caractéristiques » : distribution des HP, types, niveaux d’évolution, nombre d’attaques, variantes et retraits. Des filtres permettent de sélectionner une série, un set, une catégorie ou une rareté.
- Vue « Historique du catalogue » : comparaison du nombre de cartes observées entre les chargements, évolution du nombre de cartes par set, changements détectés dans DIM_CARD_SCD2 et suivi des versions courantes.

Les indicateurs doivent être définis directement à partir du modèle Gold. L’utilisateur doit pouvoir partir d’un indicateur global et descendre vers une série, un set puis une carte. Cette navigation rend le dashboard utile pour explorer le catalogue sans consulter directement les données brutes.


## Qualité, limites et pistes d’amélioration

Plusieurs contrôles sont nécessaires : unicité des clés, absence de cartes sans set connu, cohérence des clés étrangères, formats de dates valides, absence de valeurs négatives pour les attributs numériques lorsque la règle métier l’interdit, cohérence des compteurs de sets, unicité de la version courante SCD2 et absence de chevauchement des périodes.

La principale limite fonctionnelle est que le modèle ne décrit que ce que TCGdex expose. Il ne contient donc ni ventes, ni stock réel d’un magasin, ni données clients, ni événements opérationnels. La Gold est complète par rapport au périmètre TCGdex retenu, mais elle ne prétend pas représenter l’ensemble de l’activité d’un magasin.

Une autre limite concerne l’historisation : des chargements successifs permettent de conserver les états observés par le pipeline, mais ne permettent pas de reconstruire les changements qui auraient eu lieu entre deux observations. Si une modification est effectuée puis annulée avant le chargement suivant, elle peut ne jamais être observée.

Enfin, certains attributs sont imbriqués dans l’API. Les attaques, types, faiblesses et variantes doivent donc être normalisés avec soin pour éviter de dupliquer artificiellement les cartes dans les analyses.

À terme, le projet pourrait être enrichi par davantage de contrôles automatiques, une meilleure gestion des évolutions de schéma et une stratégie de monitoring des changements de couverture des données.


## Conclusion

Le projet transforme une API spécialisée dans le Pokémon TCG en un véritable socle décisionnel. L’architecture Bronze/Silver/Gold permet de préserver les données sources, de les nettoyer, de les historiser et de publier une Gold stable pour Power BI.

Le modèle dimensionnel proposé répond aux exigences du travail : plusieurs dimensions, une table de faits, une historisation SCD2, plusieurs loads, un reload de fact et une orchestration Databricks. Surtout, la Gold est construite exclusivement à partir des informations disponibles dans TCGdex.

Le résultat attendu est un catalogue analytique permettant de comprendre la structure des séries, des extensions et des cartes, d’explorer leurs caractéristiques et de suivre les évolutions observées entre plusieurs chargements. La valeur du projet repose donc sur la qualité de l’intégration, de l’historisation et du reporting plutôt que sur l’ajout de sources externes.
